### Practical Exercise 3B: Uniform Cost Search
#### Finding the shortest route from Nairobi to Kisumu

In this assignment, we'll be using Uniform Cost Search to find the shortest route our tractor and trailer can take from Nairobi to Kisumu.

We'll use the towns and road distances we came up with in Practical Exercise 3. As the program runs, we'll keep track of the distance travelled so far and use it to decide which town to explore next.

The idea is simple: we'll always explore the route with the lowest total distance first. We'll keep doing this until Kisumu is removed from our list of routes to explore, at which point we'll have found the shortest route in our graph.

One thing to keep in mind is that we're assuming all the roads in our graph can be travelled in both directions and that the distance is the same either way. We're also only working with the roads we included in Part 1, not every possible road in Kenya.

#### 1. Set up the problem

First, we'll enter the roads and their distances from Part 1. Each road connects two towns and has a distance attached to it.

We'll also specify where our journey starts and where we want to end up. Keeping these values in variables makes it easier to change them later if we need to.

In [1]:
#heapq helps us select the lowest-cost route waiting to be explored.
import heapq

#count gives each queue entry a unique number when two routes have the same cost.
from itertools import count

# The roads between towns.
roadConnections = [
    ("Nairobi", "Nakuru", 161),
    ("Nairobi", "Nyeri", 151),
    ("Nyeri", "Nyahururu", 104),
    ("Nyahururu", "Nakuru", 64),
    ("Nakuru", "Mau Summit", 62),
    ("Mau Summit", "Kericho", 50),
    ("Kericho", "Kisumu", 84),
    ("Nairobi", "Narok", 141),
    ("Narok", "Bomet", 81),
    ("Bomet", "Kericho", 77),
    ("Bomet", "Kisii", 83),
    ("Kisii", "Kisumu", 114),
]

# Initial state and goal state
initialState= "Nairobi"
goalState= "Kisumu"

print(f"Starting town: {initialState}")
print(f"Destination:   {goalState}")
print(f"Roads listed:  {len(roadConnections)}")

Starting town: Nairobi
Destination:   Kisumu
Roads listed:  12


#### 2. Build the road network

Before we start searching, we need a way for the program to know which towns connect to each other.

We'll use our road list to build a network that shows each town, the towns we can reach from it, and the distance we'll need to travel to get there. Since our roads work in both directions, we'll add a connection for each direction.

We'll then print the network so we can see what we're working with before the search begins.

In [2]:
def buildGraph(roadConnections):
    """Build a list of neighbouring towns and road distances for every town."""
    roadGraph = {}
    for firstTown, secondTown, roadDistance in roadConnections:
        # Add the first town if this is the first road we've seen for it.
        if firstTown not in roadGraph:
            roadGraph[firstTown] = []
        # Add the second town if this is the first road we've seen for it.
        if secondTown not in roadGraph:
            roadGraph[secondTown] = []
        # Part 1 assumes this road can be travelled in both directions.
        roadGraph[firstTown].append((secondTown, roadDistance))
        roadGraph[secondTown].append((firstTown, roadDistance))
    
    
    for town in roadGraph:
        roadGraph[town].sort(key=lambda connection:(connection[1], connection[0]))
    
    return roadGraph

roadGraph = buildGraph(roadConnections)

print("Towns in our graph:")
for town in sorted(roadGraph):
    print(f" {town}")

print("\nRoad connections (distance in km):")
for town in sorted(roadGraph):
    neighbourDescriptions = []
    for neighbourTown, roadDistance in roadGraph[town]:
        neighbourDescriptions.append(f"{neighbourTown} ({roadDistance} km)")
    print(f" {town}--->{', '.join(neighbourDescriptions)}")



Towns in our graph:
 Bomet
 Kericho
 Kisii
 Kisumu
 Mau Summit
 Nairobi
 Nakuru
 Narok
 Nyahururu
 Nyeri

Road connections (distance in km):
 Bomet--->Kericho (77 km), Narok (81 km), Kisii (83 km)
 Kericho--->Mau Summit (50 km), Bomet (77 km), Kisumu (84 km)
 Kisii--->Bomet (83 km), Kisumu (114 km)
 Kisumu--->Kericho (84 km), Kisii (114 km)
 Mau Summit--->Kericho (50 km), Nakuru (62 km)
 Nairobi--->Narok (141 km), Nyeri (151 km), Nakuru (161 km)
 Nakuru--->Mau Summit (62 km), Nyahururu (64 km), Nairobi (161 km)
 Narok--->Bomet (81 km), Nairobi (141 km)
 Nyahururu--->Nakuru (64 km), Nyeri (104 km)
 Nyeri--->Nyahururu (104 km), Nairobi (151 km)


#### 3. Create a few helper functions

Before getting into the main algorithm, we'll create a few functions to help us along the way.

These functions will handle some of the smaller tasks, like displaying headings in our output, showing the routes we're considering, retracing the route we took to reach a town, and finding the distance between two connected towns.

This way, we can keep the main search code focused on what matters: finding our way from Nairobi to Kisumu.

In [3]:
def printBanner(title, separatorCharacter="="):
    """Print a headint that makes the console output easier to scan."""
    print("\n" + separatorCharacter * 78)
    print(title)
    print(separatorCharacter * 78)

def formatFrontier(frontier, bestCosts, settledTowns):
    """Show the current best entries waiting in the priority queue."""
    liveFrontier= []

    for pathCost, insertionOrder, town in frontier:
        # A town may have an old queue entry if we later found a cheaper route.
        #Only show entries that still match the best cost we know for that town.
        if town not in settledTowns:
            if pathCost == bestCosts[town]:
                liveFrontier.append((pathCost, insertionOrder, town))
    
    # This order matches the way the queue chooses its next entry.
    liveFrontier.sort()

    if len(liveFrontier) == 0:
        return "(empty)"
    
    frontierDescriptions = []
    for pathCost, insertionOrder, town in liveFrontier:
        frontierDescriptions.append(f"{town} [cost= {pathCost} km]")
        
    return ", ".join(frontierDescriptions)

def reconstructPath(predecessors, destinationTown):
    """Follow the saved previous-town links to rebuild the route."""
    route = []
    currentTown = destinationTown
    
    while currentTown is not None:
        route.append(currentTown)
        currentTown= predecessors[currentTown]
    # We followed the route backwards thus we reverse it to get Nairobi -> destination
    route.reverse()
    return route

def getEdgeCost(roadGraph, startingTown, destinationTown):
    """Return the distance for a road between two neighbouring towns."""
    for neighbourTown, roadDistance in roadGraph[startingTown]:
        if neighbourTown == destinationTown:
            return roadDistance
    
    raise ValueError(f"No road connects {startingTown} and {destinationTown} in this graph.")

#### 4. Write the Uniform Cost Search algorithm

This is where the actual searching happens.

We'll start in Nairobi with a distance of 0 km. From there, we'll keep a list of towns we can explore and the total distance it would take to reach each one. Every time we need to choose our next town, we'll pick the one with the lowest total distance.

At each step, we'll check whether we've already dealt with that town, show how we got there, and check whether we've reached Kisumu. If we haven't, we'll look at the roads leading out of that town and calculate how much it would cost to travel to each neighbouring town.

If we find a new town, we'll add it to our list. If we find a cheaper way to reach a town we've already discovered, we'll update its distance and route. If the new route costs more than the one we already know, we'll leave things as they are.

We'll repeat this process until Kisumu is selected. Notice that we won't stop just because we've come across Kisumu; we need to make sure it's the cheapest option in our list first.

In [4]:
def uniformCostSearch(roadGraph, startTown, destinationTown):
    # The queue stores (total distance so far, tie-break number, town).
    # The tie-break number keeps equal-cost entries predictable.
    tieBreaker = count()
    frontier = [(0, next(tieBreaker), startTown)]

    # bestCosts remembers the cheapest route we've found to each town.
    bestCosts = {startTown: 0}

    # predecessors remembers the town we came from on that cheapest route.
    predecessors = {startTown: None}

    # Once a town is settled, its cheapest distance is final for this graph.
    settledTowns = set()

    # We'll keep a record of each road option considered for the final printout.
    searchTreeRecord = []
    selectedTownCount = 0
    expandedTownCount = 0

    printBanner("UNIFORM COST SEARCH")
    print(f"Start: {startTown}")
    print(f"Goal:  {destinationTown}")
    print("Starting cost is 0 km because the trailer has not travelled yet.")
    print(f"Frontier: {formatFrontier(frontier, bestCosts, settledTowns)}")
    print("Settled towns: (none yet)")
    print("\nImportant rule: we check for Kisumu when it is removed from the frontier.")
    print("Finding Kisumu as a neighbour is not enough to stop, because a cheaper")
    print("route to it might still be waiting to be found.")

    while len(frontier) > 0:
        # Remove whichever available entry has the smallest total distance.
        pathCost, insertionOrder, currentTown = heapq.heappop(frontier)

        # If this town was already settled, this queue entry can be ignored.
        if currentTown in settledTowns:
            print(f"\n[SKIP] {currentTown} at {pathCost} km was already settled.")
            continue

        # A better route may have been found after this older entry was added.
        if pathCost != bestCosts[currentTown]:
            print(f"\n[SKIP] Old entry for {currentTown}: {pathCost} km.")
            print(f"       The best known cost is now {bestCosts[currentTown]} km, so this entry is out of date.")
            continue

        selectedTownCount += 1
        printBanner(f"STEP {selectedTownCount}: TAKE {currentTown.upper()} FROM THE FRONTIER", "-")
        print(f"We selected {currentTown} because its current cost is the lowest: {pathCost} km.")
        print(f"Route so far: {' -> '.join(reconstructPath(predecessors, currentTown))}")
        print(f"Other options waiting: {formatFrontier(frontier, bestCosts, settledTowns)}")

        # This is the cheapest time to settle this town: no cheaper route can beat it now.
        settledTowns.add(currentTown)

        # Check the goal only after taking the town from the priority queue.
        if currentTown == destinationTown:
            print(f"\nGOAL TEST: Is {currentTown} the destination {destinationTown}? YES.")
            print(f"We reached the goal with a total path cost of {pathCost} km.")
            print("Because this was the lowest-cost entry in the frontier, the route is optimal for this graph.")
            print("Stopping the search here: there is no need to explore Kisumu's neighbours.")

            finalPath = reconstructPath(predecessors, destinationTown)
            return finalPath, pathCost, searchTreeRecord, selectedTownCount, expandedTownCount, settledTowns

        print(f"\nGOAL TEST: Is {currentTown} the destination {destinationTown}? No.")
        print(f"We will now check the roads leaving {currentTown}.")
        expandedTownCount += 1

        for neighbourTown, roadDistance in roadGraph[currentTown]:
            newPathCost = pathCost + roadDistance
            costCalculation = f"{pathCost} + {roadDistance} = {newPathCost} km"

            print(f"\n  Road option: {currentTown} -> {neighbourTown} ({roadDistance} km)")
            print(f"  Cost if we take this road: {costCalculation}.")

            if neighbourTown in settledTowns:
                print(f"  Decision: ignore it. We've already settled {neighbourTown} at its cheapest cost.")
                searchTreeRecord.append((currentTown, neighbourTown, newPathCost, "already settled; ignored"))

            elif neighbourTown not in bestCosts:
                # We've never reached this town before, so save this route for now.
                bestCosts[neighbourTown] = newPathCost
                predecessors[neighbourTown] = currentTown
                heapq.heappush(frontier, (newPathCost, next(tieBreaker), neighbourTown))
                searchTreeRecord.append((currentTown, neighbourTown, newPathCost, "new route"))
                print(f"  Decision: add {neighbourTown} to the frontier at {newPathCost} km. This is our first route to it.")

            elif newPathCost < bestCosts[neighbourTown]:
                # We've found a cheaper way to a town, so update its cost and previous town.
                previousBestCost = bestCosts[neighbourTown]
                bestCosts[neighbourTown] = newPathCost
                predecessors[neighbourTown] = currentTown
                heapq.heappush(frontier, (newPathCost, next(tieBreaker), neighbourTown))
                searchTreeRecord.append((currentTown, neighbourTown, newPathCost, f"improved from {previousBestCost} km"))
                print(f"  Decision: keep this route. {newPathCost} km is cheaper than the previous best of {previousBestCost} km.")
                print("  The old queue entry can stay there for now; we'll skip it if it comes up later.")

            elif newPathCost == bestCosts[neighbourTown]:
                # A tie does not improve the best cost, so we don't need another queue entry.
                searchTreeRecord.append((currentTown, neighbourTown, newPathCost, "same cost; no update"))
                print(f"  Decision: the cost ties the current best of {bestCosts[neighbourTown]} km, so no update is needed.")

            else:
                # This option gets us there, but it costs more than the route we already know.
                searchTreeRecord.append((currentTown, neighbourTown, newPathCost, "more expensive; discarded"))
                print(f"  Decision: discard it. We already know a cheaper route costing {bestCosts[neighbourTown]} km.")

        print(f"\nFrontier after checking {currentTown}: {formatFrontier(frontier, bestCosts, settledTowns)}")
        settledTownDescriptions = ", ".join(sorted(settledTowns))
        print(f"Towns settled so far: {settledTownDescriptions}")

    # If the frontier becomes empty, none of the available routes reached the goal.
    print(f"No route from {startTown} to {destinationTown} was found in this graph.")
    return None, float("inf"), searchTreeRecord, selectedTownCount, expandedTownCount, settledTowns

#### 5. Running the search

Now we search.

In [5]:
optimalPath, optimalCost, searchTreeRecord, selectedTownCount, expandedTownCount, settledTowns = uniformCostSearch(
    roadGraph,
    initialState,
    goalState
)

printBanner("SEARCH FINISHED")
if optimalPath is None:
    print("The trailer could not reach Kisumu using the roads in this graph.")
else:
    print("Optimal route:")
    print(" -> ".join(optimalPath))
    print(f"Total path cost: {optimalCost} km")
    print(f"Valid towns removed from the frontier, including Kisumu: {selectedTownCount}")
    print(f"Non-goal towns whose outgoing roads were checked: {expandedTownCount}")


UNIFORM COST SEARCH
Start: Nairobi
Goal:  Kisumu
Starting cost is 0 km because the trailer has not travelled yet.
Frontier: Nairobi [cost= 0 km]
Settled towns: (none yet)

Important rule: we check for Kisumu when it is removed from the frontier.
Finding Kisumu as a neighbour is not enough to stop, because a cheaper
route to it might still be waiting to be found.

------------------------------------------------------------------------------
STEP 1: TAKE NAIROBI FROM THE FRONTIER
------------------------------------------------------------------------------
We selected Nairobi because its current cost is the lowest: 0 km.
Route so far: Nairobi
Other options waiting: (empty)

GOAL TEST: Is Nairobi the destination Kisumu? No.
We will now check the roads leaving Nairobi.

  Road option: Nairobi -> Narok (141 km)
  Cost if we take this road: 0 + 141 = 141 km.
  Decision: add Narok to the frontier at 141 km. This is our first route to it.

  Road option: Nairobi -> Nyeri (151 km)
  Cost if 

#### 6. Show the final route and its cost

We've finished the search, so let's see what route the algorithm found.

We'll print each road segment, the distance between the two towns, and the total distance travelled up to that point. This will make it easier to follow the journey from Nairobi to Kisumu and see how we arrived at the final cost.

At the end, we'll add up the distances for every segment and display the total cost of the journey. This is our main answer to the assignment: the optimal path and its associated cost.

In [6]:
printBanner("FINAL ANSWER: OPTIMAL PATH")

if optimalPath is None:
    print("No route is available, so there is no path cost to report.")
else:
    print("Route:", " -> ".join(optimalPath))
    print()
    print(f"{'Road segment':<34}{'Distance (km)':>16}{'Cumulative (km)':>20}")
    print("-" * 70)

    cumulativeDistance = 0
    for pathIndex in range(len(optimalPath) - 1):
        startingTown = optimalPath[pathIndex]
        destinationTown = optimalPath[pathIndex + 1]
        roadDistance = getEdgeCost(roadGraph, startingTown, destinationTown)
        cumulativeDistance = cumulativeDistance + roadDistance
        roadSegment = startingTown + " -> " + destinationTown
        print(f"{roadSegment:<34}{roadDistance:>16}{cumulativeDistance:>20}")

    distanceParts = []
    for pathIndex in range(len(optimalPath) - 1):
        startingTown = optimalPath[pathIndex]
        destinationTown = optimalPath[pathIndex + 1]
        roadDistance = getEdgeCost(roadGraph, startingTown, destinationTown)
        distanceParts.append(str(roadDistance))

    print()
    print("Total path cost:", " + ".join(distanceParts), "=", cumulativeDistance, "km")
    print("The total in the table matches the cost returned by Uniform Cost Search:", cumulativeDistance == optimalCost)
    print("\nNote: the result is optimal for the selected graph in Part 1. Roads not included in that graph are not considered.")


FINAL ANSWER: OPTIMAL PATH
Route: Nairobi -> Nakuru -> Mau Summit -> Kericho -> Kisumu

Road segment                         Distance (km)     Cumulative (km)
----------------------------------------------------------------------
Nairobi -> Nakuru                              161                 161
Nakuru -> Mau Summit                            62                 223
Mau Summit -> Kericho                           50                 273
Kericho -> Kisumu                               84                 357

Total path cost: 161 + 62 + 50 + 84 = 357 km
The total in the table matches the cost returned by Uniform Cost Search: True

Note: the result is optimal for the selected graph in Part 1. Roads not included in that graph are not considered.


#### 7. Check the answer against the other routes

In this section we'll compare our answer to other possible paths that would've led to Kisumu too.

We'll list all the simple routes in our graph and calculate their total distances. A simple route is one that doesn't visit the same town more than once.

This gives us another way to check our answer. Uniform Cost Search has already found what it considers to be the cheapest route, but we'll compare it with the other possible routes to make sure none of them costs less.

Keep in mind that we're only checking the routes available in our selected graph from Part 1. We can't claim that this is the shortest route across every road in Kenya, only that it's the cheapest route in the network we've provided.

In [8]:
def findAllSimplePaths(roadGraph, currentTown, destinationTown, currentPath, currentPathCost, foundRoutes):
    """Collect every route to the destination without visiting a town twice."""
    # Add the town we're currently visiting to the route we're building.
    currentPath.append(currentTown)

    if currentTown == destinationTown:
        # Save a copy, because currentPath will change as the recursion goes back.
        foundRoutes.append((currentPath.copy(), currentPathCost))
    else:
        for neighbourTown, roadDistance in roadGraph[currentTown]:
            if neighbourTown not in currentPath:
                newPathCost = currentPathCost + roadDistance
                findAllSimplePaths(
                    roadGraph,
                    neighbourTown,
                    destinationTown,
                    currentPath,
                    newPathCost,
                    foundRoutes
                )

    # Remove this town before returning so another possible route can use the same town.
    currentPath.pop()


allSimpleRoutes = []
findAllSimplePaths(roadGraph, initialState, goalState, [], 0, allSimpleRoutes)

# Put the cheapest routes at the top of the list.
allSimpleRoutes.sort(key=lambda routeDetails: routeDetails[1])

printBanner("VERIFICATION: ALL SIMPLE ROUTES", "-")
for routePath, routeCost in allSimpleRoutes:
    routeDescription = " -> ".join(routePath)
    if routePath == optimalPath:
        print(f"{routeCost:>4} km  {routeDescription}  <-- optimal route found by UCS")
    else:
        print(f"{routeCost:>4} km  {routeDescription}")

if len(allSimpleRoutes) > 0:
    cheapestRouteCost = allSimpleRoutes[0][1]
    print()
    print(f"Cheapest simple route found by this check: {cheapestRouteCost} km")
    print(f"Uniform Cost Search result:               {optimalCost} km")
    print("Do the costs agree?", cheapestRouteCost == optimalCost)
    assert cheapestRouteCost == optimalCost, "The verification found a cheaper route than Uniform Cost Search."
    print("Check passed: the verification agrees with Uniform Cost Search.")
else:
    print("No route was found during the verification check.")

print("\nSearch tree record (road option -> cumulative cost):")
for previousTown, nextTown, newPathCost, decisionNote in searchTreeRecord:
    print(f"  {previousTown:<12} -> {nextTown:<12} cost = {newPathCost:<4} km | {decisionNote}")

print("\nConclusion: for the graph provided in Part 1, the optimal route is")
print(" -> ".join(optimalPath))
print(f"with a total path cost of {optimalCost} km.")


------------------------------------------------------------------------------
VERIFICATION: ALL SIMPLE ROUTES
------------------------------------------------------------------------------
 357 km  Nairobi -> Nakuru -> Mau Summit -> Kericho -> Kisumu  <-- optimal route found by UCS
 383 km  Nairobi -> Narok -> Bomet -> Kericho -> Kisumu
 419 km  Nairobi -> Narok -> Bomet -> Kisii -> Kisumu
 515 km  Nairobi -> Nyeri -> Nyahururu -> Nakuru -> Mau Summit -> Kericho -> Kisumu
 547 km  Nairobi -> Nakuru -> Mau Summit -> Kericho -> Bomet -> Kisii -> Kisumu
 705 km  Nairobi -> Nyeri -> Nyahururu -> Nakuru -> Mau Summit -> Kericho -> Bomet -> Kisii -> Kisumu

Cheapest simple route found by this check: 357 km
Uniform Cost Search result:               357 km
Do the costs agree? True
Check passed: the verification agrees with Uniform Cost Search.

Search tree record (road option -> cumulative cost):
  Nairobi      -> Narok        cost = 141  km | new route
  Nairobi      -> Nyeri        cost = 